In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = "/content/drive/MyDrive/Recipe recommendation system"
os.chdir(project_path)
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Recipe recommendation system


In [2]:
!pip install mlflow
!pip install loguru

In [28]:
import pandas as pd
import numpy as np
import ast
import numpy as np
import torch
import mmh3
from transformers import AutoTokenizer,DistilBertModel
import itertools
import pyarrow.parquet as pq
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import mlflow
from modules import DataFramePreprocessing, CreationDataset,CollateFunction,set_seed,seed_worker,EncodedHashedEmbeddings,Scaler,split_df,RecommendationModel,Train,BERTCreationDataset,BERTCreationDataset,BERTEmbeddingsExtractor,BertCollateFunction,ContrastiveMSELoss

In [29]:
seed=42
set_seed(seed=seed)

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Recommendation_model_training")

KeyboardInterrupt: 

In [30]:
mlflow.set_tracking_uri("file:./modules/mlflow_runs/mlruns")
mlflow.set_experiment("Recommendation_model_training")

/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/02 19:29:03 INFO mlflow.tracking.fluent: Experiment with name 'Recommendation_model_training' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///content/drive/MyDrive/Recipe recommendation '
 'system/modules/mlflow_runs/mlruns/282575499087463590'), creation_time=1775158143443, experiment_id='282575499087463590', last_update_time=1775158143443, lifecycle_stage='active', name='Recommendation_model_training', tags={}, workspace='default'>

## Creation of the processed dataframe

In [4]:
interactions_train_df=pd.read_csv("data/interactions_train.csv")
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")
RAW_interactions_df=pd.read_csv("data/RAW_interactions.csv")
RAW_recipes_df=pd.read_csv("data/RAW_recipes.csv")


In [4]:
interactions_train_df_filtered=interactions_train_df.sample(n=20000,replace=False,random_state=42)

In [5]:
print(interactions_train_df_filtered.shape)


(20000, 6)


In [6]:
RAW_recipes_df.rename(columns={"id":"recipe_id"},inplace=True)
PP_recipes_df.rename(columns={"techniques":"techniques_recipes"},inplace=True)
PP_users_df.rename(columns={"techniques":"techniques_users"},inplace=True)

In [7]:
print("interactions_train_df_filtered columns:")
print(interactions_train_df_filtered.columns)
print("------------------------------------------------------------------")
print("PP_recipes_df columns:")
print(PP_recipes_df.columns)
print("-------------------------------------------------------------------")
print("PP_users_df columns:")
print(PP_users_df.columns)
print("----------------------------------------------------------------------")
print("RAW_interactions_df columns:")
print(RAW_interactions_df.columns)
print("------------------------------------------------------------------------")
print("RAW_recipes_df columns:")
print(RAW_recipes_df.columns)

interactions_train_df_filtered columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'u', 'i'], dtype='object')
------------------------------------------------------------------
PP_recipes_df columns:
Index(['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens',
       'techniques_recipes', 'calorie_level', 'ingredient_ids'],
      dtype='object')
-------------------------------------------------------------------
PP_users_df columns:
Index(['u', 'techniques_users', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
----------------------------------------------------------------------
RAW_interactions_df columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')
------------------------------------------------------------------------
RAW_recipes_df columns:
Index(['name', 'recipe_id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dty

In [8]:
full_df=interactions_train_df_filtered.merge(PP_recipes_df,on="i",how="left",suffixes=(None,"_1")).merge(PP_users_df,on="u",how="left",suffixes=(None,"_2")).merge(RAW_interactions_df,on=["user_id","recipe_id"],how="left",suffixes=(None,"_3")).merge(RAW_recipes_df,on=["recipe_id"],how="left",suffixes=(None,"_3"))

In [9]:
full_df=full_df.drop(columns=["review",'rating_3',"id","ingredients","submitted","contributor_id","date_3","date","steps_tokens","ingredient_tokens","name_tokens","u"],axis=1)

In [10]:
print(f"Columns of the full dataframe:")
print(full_df.columns)
print("----------------------------------------")
print(f"Shape of the full dataframe:")
print(full_df.shape)
print("----------------------------------------")
print(f"Columns of the filtered interaction dataframe:")
print(interactions_train_df_filtered.shape)
print("----------------------------------------")
print(f"Head of the full  dataframe:")
display(full_df.head())
print("----------------------------------------")
for col in full_df.columns:
    print(f"Unique values of {col} :")
    print(full_df[col].unique())
    print("----------------------------------------")
for col in full_df.columns:
    print(f"The type of {col} is")
    print(full_df[col].apply(type).unique())
    print("---------------------------------")

Columns of the full dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full dataframe:
(20000, 20)
----------------------------------------
Columns of the filtered interaction dataframe:
(20000, 6)
----------------------------------------
Head of the full  dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,ratings,n_ratings,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,231160,124810,5.0,170682,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8911, 20238, 117521, 79316, 154563, 34302, 17...",7,"[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]",7,huckleberry or blueberry coffee cake,75,"['time-to-make', 'course', 'main-ingredient', ...","[174.1, 4.0, 92.0, 8.0, 7.0, 3.0, 11.0]",11,['beat margarine and cream cheese at medium sp...,"cooking light published this in their book, fi...",10
1,142629,31342,5.0,120865,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122019, 150060, 117906, 82202, 119046, 141911...",10,"[4.0, 5.0, 5.0, 5.0, 4.0, 5.0, 3.0, 5.0, 4.0, ...",10,blender quiche or whatever you have in your ...,65,"['weeknight', 'time-to-make', 'course', 'main-...","[308.4, 37.0, 8.0, 20.0, 21.0, 41.0, 3.0]",10,"['preheat oven to 350f', 'generously grease a ...",the great part is you can use whatever leftove...,12
2,822358,441841,4.0,23142,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 6270]","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29761, 87944, 72370, 139097, 39470, 37755, 23...",17,"[4.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0, 5.0, 5.0, ...",17,sweet orange coleslaw,10,"['15-minutes-or-less', 'time-to-make', 'course...","[290.1, 21.0, 71.0, 14.0, 7.0, 8.0, 13.0]",6,"['chop the apples', 'in a small bowl mix the o...",this recipe came from one of my mom's cookbook...,8
3,655579,211110,4.0,156541,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590]","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34492, 71279, 73805, 68721, 114795, 35164, 33...",28,"[2.0, 3.0, 4.0, 2.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",28,wilted greens with garlic and balsamic vinegar,15,"['15-minutes-or-less', 'time-to-make', 'course...","[77.5, 10.0, 5.0, 5.0, 2.0, 4.0, 1.0]",5,"['cut the greens into strips 1 inch wide', 'in...",this simple but delicious side dish can be mad...,7
4,35140,114469,5.0,30839,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121722, 36691, 151293, 154016, 1205, 12802, 1...",364,"[5.0, 5.0, 0.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",364,chicken in the limelight,75,"['time-to-make', 'course', 'main-ingredient', ...","[645.9, 69.0, 11.0, 22.0, 80.0, 54.0, 4.0]",8,"['preheat oven to 350', 'grate lime peel and s...",very tender chicken soaked with flavor,9


----------------------------------------
Unique values of user_id :
[231160 142629 822358 ... 241982 164867 137854]
----------------------------------------
Unique values of recipe_id :
[124810  31342 441841 ...  98278 274099 283283]
----------------------------------------
Unique values of rating :
[5. 4. 3. 2. 0. 1.]
----------------------------------------
Unique values of i :
[170682 120865  23142 ... 108495 128044  25867]
----------------------------------------
Unique values of techniques_recipes :
['[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [11]:
PP_users_df["len_items"]=PP_users_df["items"].str.count(r",")+1
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
max_len_items=PP_users_df["len_items"].max()
PP_users_df.drop(columns=["len_items"],inplace=True)
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [12]:
full_df_processed=DataFramePreprocessing(full_df,max_len_ingredients,max_len_items)
full_df_processed=full_df_processed.preprocessing()

In [13]:
PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [14]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]

In [15]:
full_df_processed["ingredient_ids_continuous"]=full_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [16]:
full_df_processed["full_text"]=full_df_processed["name"]+" [SEP] "+full_df_processed["description"]+" [SEP] "+full_df_processed["steps"]+" [SEP] "+full_df_processed["tags"]

In [17]:
scaler=StandardScaler()
full_df_processed=Scaler(scaler).scale(full_df_processed)

In [18]:
full_df_processed["rating_scaled"]=full_df_processed["rating"]/5

In [19]:
full_df_processed.to_parquet(path="./modules/dataframe/full_df_processed.parquet")

## Load of the dataframe

In [31]:
parquet_file = pq.ParquetFile("./modules/dataframe/full_df_processed.parquet")

chunks = []
for batch in parquet_file.iter_batches(batch_size=10000):
    chunks.append(batch.to_pandas())

full_df_processed = pd.concat(chunks, ignore_index=True)

In [32]:
print(f"Columns of the full processed dataframe:")
print(full_df_processed.columns)
print("----------------------------------------")
print(f"Shape of the full processed dataframe:")
print(full_df_processed.shape)
print("----------------------------------------")
print(f"Head of the full  processed dataframe:")
display(full_df_processed.head())
print("----------------------------------------")
for col in full_df_processed.columns:
    print(f"Exemple of {col} :")
    print(full_df_processed.iloc[0][col])
    print("----------------------------------------")
for col in full_df_processed.columns:
    print(f"The type of {col} is")
    print(full_df_processed[col].apply(type).unique())
    print("---------------------------------")

Columns of the full processed dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients',
       'ingredient_ids_continuous', 'full_text', 'calorie_level_scaled',
       'n_items_scaled', 'n_ratings_scaled', 'minutes_scaled',
       'n_steps_scaled', 'n_ingredients_scaled', 'rating_scaled'],
      dtype='object')
----------------------------------------
Shape of the full processed dataframe:
(20000, 29)
----------------------------------------
Head of the full  processed dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,n_ingredients,ingredient_ids_continuous,full_text,calorie_level_scaled,n_items_scaled,n_ratings_scaled,minutes_scaled,n_steps_scaled,n_ingredients_scaled,rating_scaled
0,231160,124810,5.0,170683,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8912, 20239, 117522, 79317, 154564, 34303, 17...",7,...,10,"[4560, 1904, 6880, 2493, 63, 331, 6248, 7443, ...",huckleberry or blueberry coffee cake [SEP] ...,-1.092504,-0.586471,-0.586471,-0.024165,0.561965,0.326400,1.0
1,142629,31342,5.0,120866,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122020, 150061, 117907, 82203, 119047, 141912...",10,...,12,"[1163, 1278, 7992, 2493, 329, 6248, 3209, 2312...",blender quiche or whatever you have in your ...,0.183190,-0.583351,-0.583351,-0.056047,0.306358,0.953249,1.0
2,822358,441841,4.0,23143,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 627...","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29762, 87945, 72371, 139098, 39471, 37756, 23...",17,...,8,"[1642, 1700, 3339, 5281, 3036, 4609, 2142, 624...",sweet orange coleslaw [SEP] this recipe came f...,0.183190,-0.576070,-0.576070,-0.231399,-0.716071,-0.300449,0.8
3,655579,211110,4.0,156542,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590, 0, 0,...","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34493, 71280, 73806, 68722, 114796, 35165, 33...",28,...,7,"[7005, 4990, 338, 3195, 5953, 6248, 588, 1, 1,...",wilted greens with garlic and balsamic vinegar...,-1.092504,-0.564629,-0.564629,-0.215458,-0.971678,-0.613873,0.8
4,35140,114469,5.0,30840,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121723, 36692, 151294, 154017, 1206, 12803, 1...",364,...,9,"[1235, 6248, 4990, 1252, 2677, 5151, 798, 2455...",chicken in the limelight [SEP] very tender chi...,1.458883,-0.215162,-0.215162,-0.024165,-0.204856,0.012976,1.0


----------------------------------------
Exemple of user_id :
231160
----------------------------------------
Exemple of recipe_id :
124810
----------------------------------------
Exemple of rating :
5.0
----------------------------------------
Exemple of i :
170683
----------------------------------------
Exemple of techniques_recipes :
[1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
----------------------------------------
Exemple of calorie_level :
0
----------------------------------------
Exemple of ingredient_ids :
[4574 1910 6906 2499   63  332 6270 7470 3819 3497    0    0    0    0
    0    0    0    0    0    0]
----------------------------------------
Exemple of techniques_users :
[3 0 0 0 1 0 0 0 0 2 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 2 1 0 0 0 2 0 0 1
 0 0 0 0 0 1 2 0 0 0 1 0 0 0 0 0 0 1 0 0 1]
----------------------------------------
Exemple of items :
[  8912  20239 117522 ...      0      0      0]


In [ ]:
#df_text=full_df_processed["full_text"]

In [33]:
distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#bert_dataset=BERTCreationDataset(df_text)

In [ ]:
#generator=torch.Generator()
#generator.manual_seed(seed)
#bert_collate_function_object=BertCollateFunction(tokenizer=tokenizer)
#bert_dataloader=DataLoader(dataset=bert_dataset,batch_size=32,shuffle=False,collate_fn=bert_collate_function_object.collate_fn,worker_init_fn=seed_worker,generator=generator)

In [ ]:
#device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
#bert_extractor=BERTEmbeddingsExtractor(bert_model=distilbert_model,device=device)
#bert_extractor.get_bert_embeddings(dataloader=bert_dataloader,path="./modules/BERT_embeddings")

In [34]:
all_cls_embeddings=torch.load("./modules/BERT_embeddings/all_cls_embeddings.pt")
all_mean_embeddings=torch.load("./modules/BERT_embeddings/all_mean_embeddings.pt")

In [35]:
print(all_cls_embeddings.shape)
print(all_mean_embeddings.shape)

torch.Size([20000, 768])
torch.Size([20000, 768])


## Model

In [36]:
train_df,val_df,test_df,train_cls_embeddings,val_cls_embeddings,test_cls_embeddings,train_mean_embeddings,val_mean_embeddings,test_mean_embeddings=split_df(full_df_processed,cls_embeddings=all_cls_embeddings,mean_embeddings=all_mean_embeddings,random_state=seed)

In [37]:
train_dataset=CreationDataset(train_df,cls_embeddings=train_cls_embeddings,mean_embeddings=train_mean_embeddings)
val_dataset=CreationDataset(val_df,cls_embeddings=val_cls_embeddings,mean_embeddings=val_mean_embeddings)

In [38]:
train_dataset.__getitem__(10)

{'user_id': np.int64(440299),
 'recipe_id': np.int64(5012),
 'rating': np.float64(5.0),
 'i': np.int64(109198),
 'techniques_recipes': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 'calorie_level': np.int64(1),
 'ingredient_ids': array([  63, 5006, 2499, 7168, 7655, 7956, 6906,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32),
 'techniques_users': array([5, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 1, 2,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]),
 'items': array([ 18676,  75351, 109198, ...,      0,      0,      0], dtype=int32),
 'n_items': np.int64(10),
 'ratings': array([2, 5, 5, ..., 0, 0, 0], dtype=int32),
 'n_ratings': np.int64(10),
 'name': 'chicago style deep dish pizza crust',
 '

In [39]:
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction(tokenizer=tokenizer)
collate_fn=collate_function_object.collate_fn
train_dataloader=DataLoader(train_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

In [40]:
batch=next(iter(train_dataloader))
for key in list(batch.keys()):
    print(key)
    print(f"The shape of batch {key} is :")
    print(batch[key].shape)
    print("-------------------")

user_id
The shape of batch user_id is :
torch.Size([64, 1])
-------------------
recipe_id
The shape of batch recipe_id is :
torch.Size([64, 1])
-------------------
rating_scaled
The shape of batch rating_scaled is :
torch.Size([64, 1])
-------------------
i
The shape of batch i is :
torch.Size([64, 1])
-------------------
technique_recipes
The shape of batch technique_recipes is :
torch.Size([64, 58])
-------------------
calorie_level_scaled
The shape of batch calorie_level_scaled is :
torch.Size([64, 1])
-------------------
ingredient_ids
The shape of batch ingredient_ids is :
torch.Size([64, 20])
-------------------
techniques_users
The shape of batch techniques_users is :
torch.Size([64, 58])
-------------------
items
The shape of batch items is :
torch.Size([64, 6437])
-------------------
n_items_scaled
The shape of batch n_items_scaled is :
torch.Size([64, 1])
-------------------
ratings_scaled
The shape of batch ratings_scaled is :
torch.Size([64, 6437])
-------------------
n_rat

In [ ]:
#PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
#all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
#ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [ ]:
#n_recipes_ids=len(np.unique(PP_recipes_df["i"]))
#n_ingredients_ids=len(all_ingredient_ids)

#print(f"There are {n_recipes_ids} recipe ids")
#print(f"There are {n_ingredients_ids} ingredient ids")

There are 178265 recipe ids
There are 7993 ingredient ids


In [ ]:
#hash_encoder=EncodedHashedEmbeddings(n_recipes_ids,1024,n_ingredients_ids,1024)
#hashed_recipes_ids_encoded_embeddings,hashed_ingredients_ids_encoded_embeddings=hash_encoder.get_encoded_hashed_embeddings("./modules/hashed_encoded_tables")

In [41]:
hashed_ingredients_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_ingredients.pt")
hashed_recipes_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_recipes.pt")

In [42]:
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model=RecommendationModel(hashed_ingredients_ids_encoded_embeddings=hashed_ingredients_ids_encoded_embeddings,hashed_recipes_ids_encoded_embeddings=hashed_recipes_ids_encoded_embeddings,device=device)

In [43]:
print(model)

RecommendationModel(
  (dhe_fnn_ingredient): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): Mish()
    (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (3): Linear(in_features=512, out_features=1024, bias=True)
  )
  (projection_ingredient): Linear(in_features=1024, out_features=256, bias=True)
  (norm_encoded_ingredients): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (dhe_fnn_items): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): Mish()
    (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (3): Linear(in_features=512, out_features=1024, bias=True)
  )
  (projection_items): Linear(in_features=1024, out_features=256, bias=True)
  (norm_encoded_items): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (projection_full_text): Linear(in_features=768, out_features=384, bias=True)
  (norm_full_text): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (first_norm_add_feat

In [44]:
loss_fn=ContrastiveMSELoss(alpha=0.3,temperature=0.2)
trainer=Train(train_dataloader,val_dataloader,model,device,loss_fn)

In [45]:
trainer.run_training("modules/train_savings")

2026-04-02 19:29:59.858 | INFO     | modules.train:run_training:84 - Epoch 0 : 
2026-04-02 19:32:48.210 | INFO     | modules.train:run_training:175 - Epoch 0 : train total loss = 1.1727715484592893
2026-04-02 19:32:48.211 | INFO     | modules.train:run_training:176 - Epoch 0 : train contrative loss = 3.815158455743702
2026-04-02 19:32:48.212 | INFO     | modules.train:run_training:177 - Epoch 0 : train MSE loss = 0.040319949734545904
2026-04-02 19:32:48.213 | INFO     | modules.train:run_training:179 - Epoch 0 : validation total loss = 1.1667602269545845
2026-04-02 19:32:48.214 | INFO     | modules.train:run_training:180 - Epoch 0 : validation contrastive loss = 3.7983742900516675
2026-04-02 19:32:48.215 | INFO     | modules.train:run_training:181 - Epoch 0 : validation MSE loss = 0.03892554524962021
2026-04-02 19:32:48.272 | INFO     | modules.train:run_training:84 - Epoch 1 : 
2026-04-02 19:35:36.469 | INFO     | modules.train:run_training:175 - Epoch 1 : train total loss = 1.1679205

In [29]:
batch=next(iter(train_dataloader))

In [30]:
print(batch["rating_scaled"].shape)

torch.Size([32, 1])
